# Tutorial 2 — Factorize

This tutorial covers the `scads_drvi.factorize.model` API:

- Reading a fit record with `fit_meta()`
- Inspecting checkpoint keys and detecting the `torch.compile` prefix
- Repairing a compile-prefixed checkpoint with `repair_compile_prefix()`
- `load_kwargs()` for multi-GPU jobs
- `split_responsibility()` for gated models
- Saving and reading per-cell latent loadings

We use a synthetic 3,000-cell plant dataset. The real `load_fit()` / `latent()` calls
(shown in the final markdown cell) require `scvi-tools >= 1.5`; everything else runs
with standard PyTorch and NumPy.

## 0. Shared setup — synthetic plant dataset

In [1]:
from pathlib import Path
import tempfile, json
import numpy as np
import pandas as pd
import torch
import h5py

from scads_drvi.config import Project

TISSUES  = ("leaf", "root", "stem", "flower")
LINEAGES = ("mesophyll", "epidermis", "vasculature")
FINE_TYPES = {
    "mesophyll":   ("meso_1", "meso_2", "meso_3"),
    "epidermis":   ("epi_1",  "epi_2"),
    "vasculature": ("vasc_1", "vasc_2", "vasc_3"),
}
CULTIVARS = ("cv_alpha", "cv_beta", "cv_gamma")
TRAITS    = ("plant_height", "grain_yield")
N_CELLS, N_FACTORS = 3_000, 12

root = Path(tempfile.mkdtemp())
rng  = np.random.default_rng(0)

fine        = [t for lin in LINEAGES for t in FINE_TYPES[lin]]
lineage_of  = {t: lin for lin in LINEAGES for t in FINE_TYPES[lin]}
cell_ids    = np.array([f"plate{i % 5}:bc{i:05d}".encode() for i in range(N_CELLS)])
fine_codes  = rng.integers(0, len(fine),      size=N_CELLS).astype(np.int32)
tissue_codes= rng.integers(0, len(TISSUES),   size=N_CELLS).astype(np.int32)
cult_codes  = rng.integers(0, len(CULTIVARS),  size=N_CELLS).astype(np.int32)
reads       = rng.integers(500, 200_000,        size=N_CELLS).astype(np.int64)

obs_path = root / "matrix.h5ad"
with h5py.File(obs_path, "w") as fh:
    obs = fh.create_group("obs")
    obs.attrs["_index"] = "cell_uid"
    obs.attrs["encoding-type"] = "dataframe"
    obs.create_dataset("cell_uid", data=cell_ids)
    for name, codes, cats in (
        ("fine_type",  fine_codes,   fine),
        ("tissue",     tissue_codes, TISSUES),
        ("cultivar",   cult_codes,   CULTIVARS),
    ):
        g = obs.create_group(name)
        g.create_dataset("categories", data=np.array([c.encode() for c in cats]))
        g.create_dataset("codes", data=codes)
    obs.create_dataset("total_reads", data=reads)
    lin_codes = np.array(
        [LINEAGES.index(lineage_of[fine[c]]) for c in fine_codes], dtype=np.int32
    )
    g = obs.create_group("lineage")
    g.create_dataset("categories", data=np.array([l.encode() for l in LINEAGES]))
    g.create_dataset("codes", data=lin_codes)

proj = Project(root=root, fit="assembly_v3", traits=TRAITS)
print("Project root:", proj.root)

Project root: /tmp/tmp477k60i1


## 1. Write `fit.meta.json` and read it back with `fit_meta()`

The training run writes this file to record hyperparameters. `fit_meta()` reads it and
returns a typed `FitMeta` dataclass. The only required key is `n_latent`; the rest have
defaults or are inferred.

In [2]:
from scads_drvi.factorize.model import fit_meta, FitMeta

fit_dir = proj.fit_dir()
fit_dir.mkdir(parents=True, exist_ok=True)

meta_record = {
    "n_latent":        N_FACTORS,
    "n_split_latent":  4,
    "batch_key":       "cultivar",
    "min_fragment":    None,
    "depth_col":       "total_reads",
    "n_cells":         N_CELLS,
    "n_features":      50_000,
    "gene_likelihood": "zinb",
    "dispersion":      "gene",
    "seed":            42,
    "smoke_test":      False,
}
(fit_dir / "fit.meta.json").write_text(json.dumps(meta_record, indent=2))

meta = fit_meta(proj)
print("n_latent:      ", meta.n_latent)
print("n_split_latent:", meta.n_split_latent)
print("batch_key:     ", meta.batch_key)
print("is_gated:      ", meta.is_gated)   # False — no min_fragment set
print("n_cells:       ", meta.n_cells)
print("n_features:    ", meta.n_features)

n_latent:       12
n_split_latent: 4
batch_key:      cultivar
is_gated:       False
n_cells:        3000
n_features:     50000


## 2. Create a minimal checkpoint and inspect its keys

`checkpoint_keys()` reads key names from a saved `.pt` file without instantiating the
model — useful for validation on a login node where loading the full network would OOM.

We create a synthetic state dict that mirrors the structure of a real DRVI checkpoint.

In [3]:
from scads_drvi.factorize.model import checkpoint_keys, has_compile_prefix, COMPILE_PREFIX

model_dir = fit_dir / "model"
model_dir.mkdir(exist_ok=True)

# Synthetic state dict — same key structure as a real DRVI checkpoint
state_dict = {
    "encoder.layer.0.weight": torch.randn(64, 50_000),
    "encoder.layer.0.bias":   torch.zeros(64),
    "z_mean_encoder.weight":  torch.randn(N_FACTORS, 64),
    "z_mean_encoder.bias":    torch.zeros(N_FACTORS),
    "decoder.layer.0.weight": torch.randn(64, N_FACTORS),
    "decoder.layer.0.bias":   torch.zeros(64),
}
torch.save({"model_state_dict": state_dict}, model_dir / "model.pt")

keys = checkpoint_keys(model_dir)
print(f"Checkpoint has {len(keys)} keys:")
for k in keys:
    print("  ", k)

print("\nhas_compile_prefix:", has_compile_prefix(model_dir))

Checkpoint has 6 keys:
   encoder.layer.0.weight
   encoder.layer.0.bias
   z_mean_encoder.weight
   z_mean_encoder.bias
   decoder.layer.0.weight
   decoder.layer.0.bias

has_compile_prefix: False


## 3. Detect and repair a `torch.compile` prefix

When a model is saved after `torch.compile()` every key acquires an `_orig_mod.` prefix.
The checkpoint cannot be loaded without the wrapper class; `repair_compile_prefix`
writes a fixed copy to a separate directory (never overwrites the original).

In [4]:
from scads_drvi.factorize.model import repair_compile_prefix

# Create a checkpoint that looks like it was saved from a compiled module
compiled_dir = fit_dir / "model_compiled"
compiled_dir.mkdir(exist_ok=True)
compiled_sd = {f"{COMPILE_PREFIX}{k}": v for k, v in state_dict.items()}
torch.save({"model_state_dict": compiled_sd}, compiled_dir / "model.pt")

print("Before repair — first two keys:")
for k in checkpoint_keys(compiled_dir)[:2]:
    print("  ", k)
print("has_compile_prefix:", has_compile_prefix(compiled_dir))

# Repair: strip the prefix and write to a new directory
repaired_dir = fit_dir / "model_repaired"
repair_compile_prefix(compiled_dir, out_dir=repaired_dir)

print("\nAfter repair — first two keys:")
for k in checkpoint_keys(repaired_dir)[:2]:
    print("  ", k)
print("has_compile_prefix:", has_compile_prefix(repaired_dir))

Before repair — first two keys:
   _orig_mod.encoder.layer.0.weight
   _orig_mod.encoder.layer.0.bias
has_compile_prefix: True

After repair — first two keys:


   encoder.layer.0.weight
   encoder.layer.0.bias
has_compile_prefix: False


## 4. `load_kwargs()` — device pinning for multi-GPU jobs

In a `torchrun` job each rank must load its shard of the model onto its own GPU.
`load_kwargs(rank)` returns the right map-location kwargs for `DRVI.load()`. With
`rank=None` it defers to `"cuda"` if a GPU is available or `"cpu"` otherwise.

In [5]:
from scads_drvi.factorize.model import load_kwargs

kwargs = load_kwargs(rank=None)
print("load_kwargs (single-process):", kwargs)

# In a real torchrun job:
# from scads_drvi.factorize.multigpu import init
# r = init()
# kwargs = load_kwargs(r.rank)
# model = DRVI.load(repaired_dir, adata=adata, **kwargs)

load_kwargs (single-process): {}


## 5. `split_responsibility()` — gate weights for gated models

Split-latent DRVI trains a two-expert gate: each cell's contribution to a split
dimension is controlled by a Softmax over two logits. `split_responsibility` converts
those logits to responsibility weights in [0, 1].

The `is_gated` flag on `FitMeta` tells you whether the model was trained with gating.

In [6]:
from scads_drvi.factorize.model import split_responsibility

n_split = meta.n_split_latent    # 4 in our synthetic meta
logits  = torch.randn(N_CELLS, n_split, 2)

resp = split_responsibility(logits)
# Output shape equals input shape: softmax is over split_dim=-2 (n_splits),
# so each (cell, feature) position holds responsibility weights across splits.
print("logits shape:        ", tuple(logits.shape))   # (3000, 4, 2)
print("responsibility shape:", resp.shape)              # (3000, 4, 2)
print(f"range: [{float(resp.min()):.3f}, {float(resp.max()):.3f}]")
# Sum across splits must equal 1 for every (cell, feature) pair
print("split weights for cell 0, feature 0 (sum to 1):", resp[0, :, 0].round(4))

logits shape:         (3000, 4, 2)
responsibility shape: (3000, 4, 2)
range: [0.002, 0.970]
split weights for cell 0, feature 0 (sum to 1): [0.404 0.221 0.17  0.205]


## 6. Save and load latent loadings

In a real run `load_fit()` + `latent()` produce the loadings. Here we generate
synthetic ones directly. The `.npz` layout is identical to what the pipeline writes,
and `read_loadings()` from `scads_drvi.io.artifacts` reads either `.npz` or `.tsv`.

In [7]:
from scads_drvi.io.artifacts import read_loadings

arm = "assembly_v3_topfrac"
arm_dir = proj.enrich_dir(arm)
arm_dir.mkdir(parents=True, exist_ok=True)

dims     = [f"dim_{j}" for j in range(N_FACTORS)]
loadings = np.clip(rng.normal(0.4, 0.5, size=(N_CELLS, N_FACTORS)), 0, None).astype(np.float32)

npz_path = fit_dir / "topic_loadings.npz"
np.savez(
    npz_path,
    cells=np.array([c.decode() for c in cell_ids], dtype=object),
    factors=np.array(dims),
    loadings=loadings,
)
print("Saved:", npz_path)

loaded = read_loadings(npz_path, npz=npz_path, dims=dims[:6])
print("Shape (cells × 6 dims):", loaded.shape)
loaded.head()

Saved: /tmp/tmp477k60i1/data/factorize/assembly_v3/topic_loadings.npz
Shape (cells × 6 dims): (3000, 6)


,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5
plate0:bc00000,0.071698,0.975539,0.000000,0.185052,0.675476,0.613724
plate1:bc00001,0.110849,0.371965,0.380922,0.778645,0.000000,0.836112
plate2:bc00002,0.605548,0.658947,1.969110,0.000000,0.000000,1.376043
plate3:bc00003,0.181105,1.087749,0.159568,0.339762,0.303242,1.317101
plate4:bc00004,0.227869,0.000000,0.387293,0.000000,0.465158,0.000000


## Calling `load_fit()` and `latent()` with a real model

Everything above runs without a DRVI model installed. The full call sequence with a
real checkpoint is:

```python
import anndata as ad
from scads_drvi.factorize.model import (
    fit_meta, has_compile_prefix, repair_compile_prefix,
    load_fit, setup_anndata_like, latent,
)

meta  = fit_meta(proj)
adata = ad.read_h5ad(proj.data / "matrix.h5ad", backed="r")
setup_anndata_like(adata, meta)           # registers obs columns the model expects

model_dir = proj.fit_dir() / "model"
if has_compile_prefix(model_dir):
    model_dir = repair_compile_prefix(model_dir, out_dir=proj.root / "model_repaired")

model = load_fit(proj, adata=adata, rank=None, device="cpu")

z = latent(model, adata=adata, indices=None, batch_size=4096)
# z: np.ndarray of shape (n_cells, n_latent)
```

Requires `scvi-tools >= 1.5` and the matching DRVI model class.